<a href="https://colab.research.google.com/github/emilsar/NLP-TCGA/blob/main/notebooks/Fall2026/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Building the Dataset

This lab works with two files, and the whole point is that neither is any use alone:

    corpus/TCGA_Reports.csv                       9,523 rows   the report text
    cancer_type/tcga_patient_to_cancer_type.csv  11,160 rows   the cancer type

`TCGA_Reports.csv` holds 9,523 cancer pathology reports, one per patient, and says
nothing about what any of them is. `tcga_patient_to_cancer_type.csv` says which cancer
each patient has, and contains not one word of report text. To teach a computer to read
a report we need the report and its answer sitting side by side, on the same row.

That is the whole lab. Read those two files, glue them together on the patient barcode,
and look at what we ended up with. Near the end a third, much smaller file
(`cancer_type/tcga-tumor-types.csv`, 37 rows) joins in to spell codes like `BRCA` out in
full.

Nothing needs downloading by hand — pandas reads all three straight from the course
repository over the web.

**Puzzles.** Cells marked **PUZZLE** have blanks written as `___`. Fill them in and run
the cell. Each puzzle ends with an `assert`, which stays silent when you are right and
stops the cell when you are wrong. Every puzzle points at the appendix section that
covers it.

## Setup

In [ ]:
# pandas reads a CSV straight from a web address, so there is nothing to install
# and nothing to download by hand. This works the same in Colab and on your laptop.
import pandas as pd

BASE = "https://raw.githubusercontent.com/emilsar/NLP-TCGA/main/data"

## The reports

One row per patient. The file is about 35 MB, so the next cell takes a few seconds.

In [ ]:
reports = pd.read_csv(f"{BASE}/corpus/TCGA_Reports.csv")

# .shape is (rows, columns).
reports.shape

In [ ]:
# Always look at data before doing anything to it.
reports.head()

In [ ]:
# One whole report, so you know what we are really working with.
print(reports.loc[0, "text"][:400])

## The barcode

`patient_filename` is two things stuck together with a dot: the patient's barcode, then
a long random report id.

    TCGA-BP-5195.25c0b433-5557-4165-922e-2c1eac9c26f0
    ^^^^^^^^^^^^
    this part identifies the patient

The barcode is the only thing the two files have in common, so everything below depends
on getting it out.

In [ ]:
# PUZZLE 1 — cut the filename at the dot and keep the piece in front.
# Appendix A, "Splitting and joining".

reports["patient_id"] = reports["patient_filename"].apply(lambda x: x.split(___)[___])

assert reports.loc[0, "patient_id"] == "TCGA-BP-5195"
reports["patient_id"].head()

In [ ]:
# PUZZLE 2 — a barcode is only safe to match on if no patient appears twice.
# Appendix B, "Checking for duplicates". Two method names, in order.

assert not reports["patient_id"].___().___()
print("One row per patient — safe to match on.")

## The labels

The cancer type is not written in the report file. It comes from TCGA's clinical
records — a separate file, from a separate source, covering a different set of patients.

In [ ]:
labels = pd.read_csv(f"{BASE}/cancer_type/tcga_patient_to_cancer_type.csv")

labels.head()

In [ ]:
# 11,160 patients here against 9,523 reports. The extras are simply never matched.
labels.shape

## The join

A DataFrame's **index** is its row labels. Label both tables by barcode and `.loc` can
pull the right label for each report, however differently the two files happen to be
ordered.

In [ ]:
# PUZZLE 3 — label both tables by barcode, then look the labels up.
# Appendix B, "The index, and joining without merge".

reports.index = reports["patient_id"].values
labels.index = labels[___].values

reports["cancer_type"] = labels.loc[___, "cancer_type"]

assert reports["cancer_type"].notna().all()
reports[["patient_id", "cancer_type"]].head()

`BRCA` and `KIRC` are TCGA's shorthand. A third file spells them out.

In [ ]:
# Note sep=";" — this file is semicolon-separated. Always check the separator.
names = pd.read_csv(f"{BASE}/cancer_type/tcga-tumor-types.csv", sep=";")

names.head()

In [ ]:
# PUZZLE 4 — pair the two columns into a lookup: code -> full name.
# Appendix A, "Building and inverting dictionaries". Use the column names above.

code_to_name = dict(zip(names[___], names[___]))

assert code_to_name["KIRC"] == "Kidney renal clear cell carcinoma"
print(code_to_name["BRCA"])

In [ ]:
# Every code in our data must exist in the lookup, or the next line would fail
# partway through.
assert reports["cancer_type"].isin(code_to_name).all()

reports["cancer_type_name"] = reports["cancer_type"].apply(lambda c: code_to_name[c])
reports[["patient_id", "cancer_type", "cancer_type_name"]].head()

## How long is a report?

Worth knowing before choosing a model: some methods have a hard limit on how much text
they can read at once.

In [ ]:
# PUZZLE 5 — count the words in each report. Two blanks: a built-in function,
# and a string method. Appendix B, "Assignment creates a column"; Appendix A, .split().

reports["n_words"] = reports["text"].apply(lambda t: ___(t.___()))

assert reports["n_words"].sum() > 5_000_000
reports["n_words"].describe().round(1)

Half the reports are under ~430 words, but the longest runs to a few thousand. Keep that
spread in mind later.

## What are we up against?

Before building anything, find the score you have to beat: how often you would be right
by ignoring the text entirely and always guessing the most common cancer type. A model
that cannot beat this has learned nothing. Make this the first thing you compute, every
time.

In [ ]:
# PUZZLE 6 — count the classes, then get the largest share.
# Appendix B, "value_counts".

counts = reports["cancer_type"].___()
print(counts.head())
print("Number of classes:", len(counts))

baseline = reports["cancer_type"].value_counts(___=True).max()
print("Majority-class baseline:", round(baseline, 3))

In [ ]:
counts.plot(kind="bar", figsize=(10, 3), title="Reports per cancer type")

**Read the printout and the chart before moving on.**

- Which cancer type is the most common, and what fraction of the corpus is it?
  Puzzle 4's lookup gives you its full name.
- Guessing at random among 32 classes would be right about 1 in 32 times — 3.1%.
  The baseline you just computed is higher than that. Why?
- Look at the right-hand end of the bar chart. What will those classes do to a model?

## A closer look

We have a table. That is not the same as having a table worth training on. Before Lab 2
turns any of this into numbers, look at what actually came out of the join — quietly
broken rows are far cheaper to find now than after a model has learned from them.

In [ ]:
# Start with the frame as a whole: 9,523 rows, five columns, and nothing missing.
# A missing value anywhere here would mean the join silently failed for that patient.
print(reports.shape)
reports[["patient_id", "text", "cancer_type", "cancer_type_name", "n_words"]].isna().sum()

### How long are the reports, really?

`.describe()` gave you the numbers. A picture gives you the shape, and the shape is the
part that matters here.

In [ ]:
# bins=50 slices the range into 50 bars. Try a different number and see what changes.
reports["n_words"].plot(
    kind="hist", bins=50, figsize=(10, 3), title="Words per report"
)

A long tail to the right, and a pile-up against the left edge. That left edge is the
interesting part: a pathology report of twenty words is not a short report, it is a
failed one — these were scanned and OCR'd, and some pages came through as nearly
nothing.

In [ ]:
# PUZZLE 7 — how many reports are under 50 words?
# Appendix B, "Filtering rows". The comparison goes in the blank.

n_tiny = (reports[___] < 50).sum()

assert n_tiny == 362
print(n_tiny, "reports are under 50 words")

In [ ]:
# Sort by length and the worst offenders come to the top.
# Appendix B, "describe and sort_values".
shortest = reports.sort_values("n_words").head(5)
shortest[["patient_id", "cancer_type", "n_words"]]

In [ ]:
# The single shortest "report" in the corpus, printed in full.
print(repr(reports.loc[reports["n_words"].idxmin(), "text"]))

One character and a full stop. That row still carries a cancer-type label, so a model
will dutifully try to learn from it. Nobody removed these for you; deciding what to do
about them is a real decision you will have to make.

In [ ]:
# PUZZLE 8 — the same text can appear twice under different barcodes.
# Appendix B, "Checking for duplicates". One column, one method.

n_dupes = reports[___].___().sum()

assert n_dupes == 18
print(n_dupes, "reports are byte-for-byte copies of another report")

### Does length give the answer away?

`.groupby("cancer_type")` splits the table into one group per cancer type and computes
whatever you ask for inside each group — here, the median report length. It is the one
new idiom in this section.

In [ ]:
by_type = reports.groupby("cancer_type")["n_words"].median().sort_values()
by_type.plot(kind="bar", figsize=(10, 3), title="Median words per report, by cancer type")

Melanoma reports run to about 100 words; bladder reports to about 1,000 — a tenfold
spread. So report *length alone* carries real information about cancer type, before a
single word is read. Worth remembering in Lab 2: if a model does well, make sure it is
reading the text and not just measuring it.

In [ ]:
# And the other end of the bar chart from earlier: the classes with almost no data.
reports["cancer_type"].value_counts().tail(8)

The rarest cancer type has 43 reports, against 1,034 for the most common. Split those 43
into train/validation/test and you are asking a model to learn a category from a couple
of dozen examples. Expect it to fail on exactly these, and expect overall accuracy to
hide that.

## Save it

Text and answer on the same row, plus the three columns you built on top. Write it out
so you can see the thing you made as a file.

In [ ]:
reports[["patient_id", "text", "cancer_type", "cancer_type_name", "n_words"]].to_csv(
    "lab1_dataset.csv", index=False
)

# On Colab this file lives in the session only. Download it from the file browser on the
# left if you want a copy — but nothing later depends on your keeping it. Lab 2 opens
# the same source files and rebuilds this table in its first few lines, so it starts
# from scratch whether or not you still have this.
pd.read_csv("lab1_dataset.csv").head()

## What you did

- Read three files that were never designed to be used together.
- Found the one column they share, and checked it was safe to trust before trusting it.
- Joined them on it, so every report now carries its own answer.
- Measured the thing you have to beat.
- Looked hard enough at the result to find its problems: near-empty reports, exact
  duplicates, classes with 43 examples, and a length signal that leaks the answer.

Not one line of that was machine learning, and none of it was optional. Next lab: turn
the text into numbers.